In [3]:
import gdown 
url = "https://drive.google.com/file/d/1XR20SZvD4wkTN7f_BO0UTfjE6TnT-P7W/view?usp=sharing"
file_id = url.split('/')[-2]

In [4]:
#Get the common URL for downloading the file (If add the ID i got from prev code to this URl , the perticular file will be downloaded)
prefix_url = "https://drive.google.com/uc?/export=download&id="
gdown.download(prefix_url+file_id, "kideny-ct-scan-data.zip")

Downloading...
From (original): https://drive.google.com/uc?/export=download&id=1XR20SZvD4wkTN7f_BO0UTfjE6TnT-P7W
From (redirected): https://drive.google.com/uc?%2Fexport=download&id=1XR20SZvD4wkTN7f_BO0UTfjE6TnT-P7W&confirm=t&uuid=0e6c98df-b8d1-42f1-8442-cde7e70d02c9
To: c:\Users\Dell\Downloads\Kidney_Disease_Classification\research\kideny-ct-scan-data.zip
100%|██████████| 1.63G/1.63G [17:43<00:00, 1.53MB/s] 


'kideny-ct-scan-data.zip'

Data Ingestion 

In [2]:
import os
os.chdir("../") #change the to the parent dir above the child dir level 
%pwd #to check current dir
#c:\\Users\\Dell\\Downloads\\Kidney_Disease_Classification

'c:\\Users\\Dell\\Downloads\\Kidney_Disease_Classification'

In [36]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:Path
    source_URL:str
    local_data_file:Path
    unzip_dir:Path

In [37]:
CONFIG_FILE_PATH = Path("config/config.yaml")
CONFIG_FILE_PATH

WindowsPath('config/config.yaml')

In [38]:
from Kidney_Disease_classification.utils.common import read_yaml,create_directories

class CofigurationManager:
    def __init__(self,config_filepath = Path("config/config.yaml"), params_filepath =Path("params.yaml")):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        
        create_directories([self.config.artifacts_root])
    
    def get_data_ingestion_cofig(self)-> DataIngestionConfig:
        config = self.config.data_ingestion 
        create_directories([config.root_dir])
        
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir = config.unzip_dir
        )   
        return data_ingestion_config

In [39]:
import os
import zipfile
import gdown 
from Kidney_Disease_classification import logger
from Kidney_Disease_classification.utils.common import get_size
 
class DataIngestion:
    def __init__(self,config:DataIngestionConfig):
        self.config = config
        
    def download_file(self)->str:
        try:
            dataset_url = self.config.source_URL
            zip_download_dir = self.config.local_data_file
            os.makedirs("artifacts/data_ingestion",exist_ok=True)
            logger.info(f"Downloading data from {dataset_url} into file {zip_download_dir}")
            
            file_id = dataset_url.split("/")[-2]
            prefix = "https://drive.google.com/uc?/export=download&id="
            gdown.download(prefix+file_id,zip_download_dir)
            
            logger.info(f"Downloaded data from {dataset_url} into file {zip_download_dir}")
        
        except Exception as e:
            raise e 
        
    def extract_zip_file(self):
        """
        zip_file_path : str
        Extracts the zip file into the data directory
        Function returns None 
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path,exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file , 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        

In [40]:
try:
    config = CofigurationManager()
    data_ingestion_config = config.get_data_ingestion_cofig()
    data_ingestion = DataIngestion(config = data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e 

[2026-04-11 22:07:32,908: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-11 22:07:32,933: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-11 22:07:32,940: INFO: common: created directory at: artifacts]
[2026-04-11 22:07:32,943: INFO: common: created directory at: artifacts/data_ingestion]
[2026-04-11 22:07:32,948: INFO: 2176214351: Downloading data from https://drive.google.com/file/d/1XR20SZvD4wkTN7f_BO0UTfjE6TnT-P7W/view?usp=sharing into file artifacts/data_ingestion/data.zip]


Downloading...
From (original): https://drive.google.com/uc?/export=download&id=1XR20SZvD4wkTN7f_BO0UTfjE6TnT-P7W
From (redirected): https://drive.google.com/uc?%2Fexport=download&id=1XR20SZvD4wkTN7f_BO0UTfjE6TnT-P7W&confirm=t&uuid=7807113f-8a2c-40f1-aa74-9bfa984964fb
To: c:\Users\Dell\Downloads\Kidney_Disease_Classification\artifacts\data_ingestion\data.zip
100%|██████████| 1.63G/1.63G [11:26<00:00, 2.37MB/s] 


[2026-04-11 22:19:03,313: INFO: 2176214351: Downloaded data from https://drive.google.com/file/d/1XR20SZvD4wkTN7f_BO0UTfjE6TnT-P7W/view?usp=sharing into file artifacts/data_ingestion/data.zip]


In [29]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir : Path
    base_model_path : Path
    updated_base_model_path : Path
    params_image_size : list
    params_learning_rate : float
    params_include_top : bool
    params_weights :str
    params_classes : int

In [30]:
from Kidney_Disease_classification.utils.common import read_yaml,create_directories

class CofigurationManager:
    def __init__(self,config_filepath = Path("config/config.yaml"), params_filepath =Path("params.yaml")):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        create_directories([self.config.artifacts_root])
    
    def get_prepare_base_model_config(self)->PrepareBaseModelConfig:
        config = self.config.prepare_base_model
        create_directories([config.root_dir])
        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(config.base_model_path),
            updated_base_model_path=Path(config.updated_base_model_path),
            params_image_size=self.params.IMAGE_SIZE,
            params_learning_rate=self.params.LEARNING_RATE,
            params_include_top=self.params.INCLUDE_TOP,
            params_weights=self.params.WEIGHTS,
            params_classes=self.params.CLASSES
        )
        return prepare_base_model_config 


In [31]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf 

class PrepareBaseModel:
    def __init__(self , config:PrepareBaseModelConfig):
        self.config = config
        
    def get_base_model(self):
        self.base_model = tf.keras.applications.vgg16.VGG16(
            input_shape = self.config.params_image_size,
            weights = self.config.params_weights,
            include_top = self.config.params_include_top
        )
        self.save_model(path=self.config.base_model_path,model=self.base_model)
    
    @staticmethod
    def _prepare_full_model(model,classes,freeze_all,freeze_till,learning_rate):
        if freeze_all:
            for layer in model.layers:
                model.trainable = False
        elif (freeze_till is not None) and (freeze_till > 0):
            for layer in model.layers[:-freeze_till]:
                model.trainable = False
        flatten_in = tf.keras.layers.Flatten()(model.output)
        prediction = tf.keras.layers.Dense(
            units=classes,
            activation="softmax"
        )(flatten_in)
        full_model = tf.keras.models.Model(
            inputs = model.input,
            outputs = prediction
        )
        full_model.compile(
            optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
            loss = tf.keras.losses.CategoricalCrossentropy(),
            metrics=["accuracy"]
        )
        full_model.summary()
        return full_model
    
    def update_base_model(self):
        self.full_model = self._prepare_full_model(
            model = self.base_model,
            classes = self.config.params_classes,
            freeze_all=True,
            freeze_till=None,
            learning_rate=self.config.params_learning_rate
        )
        self.save_model(path=self.config.updated_base_model_path,model=self.full_model)
    
    @staticmethod
    def save_model(path:Path , model:tf.keras.Model):
        model.save(path)           

In [32]:
try:
    config = CofigurationManager()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config = prepare_base_model_config)
    prepare_base_model.get_base_model()
    prepare_base_model.update_base_model()
except Exception as e:
    raise e

[2026-04-14 12:18:27,163: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-14 12:18:27,195: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-14 12:18:27,206: INFO: common: created directory at: artifacts]
[2026-04-14 12:18:27,212: INFO: common: created directory at: artifacts/prepare_base_model]
[2026-04-14 12:18:29,085: WARNING: saving_api: You are saving your model as an HDF5 file via `model.save()` or `keras.saving.save_model(model)`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')` or `keras.saving.save_model(model, 'my_model.keras')`. ]


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 25088)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │       100,356 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,815,044 (56.51 MB)

 Trainable params: 100,356 (392.02 KB)

 Non-trainable params: 14,714,688 (56.13 MB)